In [1]:
import napari
from numpy import spacing
from skimage.io import imread
import glob
from wip_util import zero_pad_index
import numpy as np

start_iter = 645
end_iter = 647
chunk_string = f"{zero_pad_index(start_iter, 4)}-{zero_pad_index(end_iter, 4)}"
frames_per_iter = 250

start_frame = start_iter * frames_per_iter
end_frame = end_iter * frames_per_iter

frames_path = r'E:\Cryo-PALM Data for Brian\James processing ASCII and RAW Data files\PALM_Run2_slab_0001_488nm_Frames_(Processed_Slab)'


c:\Users\bnort\miniconda3\envs\decode_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def load_frames(id):
    print(f"loading frames for id: {id}")
    frames_pattern = rf'{frames_path}\3dpalm488nm_iter_{zero_pad_index(id, 4)}_0001_ch0_cam1_stack0000_405nm_0000000msec_*msecabs_000x_000y_000z_0001t.tif'
    frames_name = glob.glob(frames_pattern)
    if frames_name:
        return imread(frames_name[0])
    else:
        print(f"no frames found for id: {id}")
        return None

frames = np.concatenate([load_frames(i) for i in range(start_iter, end_iter + 1)], axis=0)

print(f"Loaded frames shape: {frames.shape}")

loading frames for id: 645
loading frames for id: 646
loading frames for id: 647
Loaded frames shape: (750, 800, 800)


In [3]:
print(frames.shape)

(750, 800, 800)


In [4]:
#frames = frames[:,:,::-1]
viewer = napari.Viewer()
viewer.add_image(frames, name='frames')

<Image layer 'frames' at 0x18fa24f67f0>

In [5]:
import pandas as pd
from torch import chunk
import decode

In [6]:
hesslab_input_file = r'E:\Cryo-PALM Data for Brian\James processing ASCII and RAW Data files\2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.txt'
hesslab_input_file = r'C:\Users\bnort\work\Janelia_slm\data\James processing ASCII and RAW Data files\2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.txt'
hesslab_input_file = r'C:\Users\bnort\work\Janelia_slm\data\James processing ASCII and RAW Data files\2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.hdf5'

In [8]:
csv = False
if csv:
    points = pd.read_csv(hesslab_input_file, sep='\t')
else:
    hesslab_emitters=decode.EmitterSet.load(hesslab_input_file)
    print(hesslab_emitters)

EmitterSet
::num emitters: 41570847
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 289749
::spanned volume: [ 2.32811e-01  1.93751e-01 -8.00000e+02] - [800.    799.475 600.   ]


In [76]:
csv = False

if csv:
    import pandas as pd

    print(f'using frames {start_frame} to {end_frame}')
    points_frame = points[(points['Frame Number'] >= start_frame) & (points['Frame Number'] < end_frame)]
    print(points.shape, points_frame.shape)
    points_np = points_frame[['Frame Number','Y Position','X Position']].to_numpy()
    #points['X Position'] = 800 - points['X Position']
    print(points_np[0,...])
    points_np[:,0] -= start_frame
    print(points_np[9000:9010,...])
    viewer.add_points(points_np, name='Hess lab method', face_color='blue', size=5)
else:
    print(hesslab_emitters.frame_ix)
    # filter based on start and end frame
    hesslab_emitters = hesslab_emitters[(hesslab_emitters.frame_ix >= start_frame) & (hesslab_emitters.frame_ix < end_frame)]
    print(hesslab_emitters.frame_ix)

tensor([     0,      0,      0,  ..., 289749, 289749, 289749])
tensor([161250, 161250, 161250,  ..., 162499, 162499, 162499])


In [77]:
print(hesslab_emitters)

EmitterSet
::num emitters: 124623
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 162499
::spanned volume: [   6.42828    1.54923 -799.976  ] - [799.946 796.218 599.985]


In [108]:

decode_input_file = rf'D:\Janelia_slm_data\Data_2025_05_27\network_CMOS_C13440_20CUd_05-01_chunk_{chunk_string}.csv'
print(start_frame, end_frame)
decode_emitters=decode.EmitterSet.load(decode_input_file)
print(decode_emitters)

161250 162500


c:\users\bnort\work\janelia_slm\code\decode\decode\generic\emitter.py:280: UserWarning: For .csv files, implicit usage of .load() is discouraged. Please use 'decode.utils.emitter_io.load_csv' explicitly.
  warnings.warn("For .csv files, implicit usage of .load() is discouraged. "


EmitterSet
::num emitters: 783247
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 162749
::spanned volume: [-4.1167417e-01 -5.7897305e-01 -1.1264720e+03] - [ 798.91437  798.9928  1039.0311 ]


In [109]:
print(decode_emitters)
decode_emitters.frame_ix = decode_emitters.frame_ix - start_frame 
print(decode_emitters)

EmitterSet
::num emitters: 783247
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 162749
::spanned volume: [-4.1167417e-01 -5.7897305e-01 -1.1264720e+03] - [ 798.91437  798.9928  1039.0311 ]
EmitterSet
::num emitters: 783247
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 1499
::spanned volume: [-4.1167417e-01 -5.7897305e-01 -1.1264720e+03] - [ 798.91437  798.9928  1039.0311 ]


In [110]:
from wip_util import get_np_points, reverse_emitters

reverse_emitters(decode_emitters, frames.shape[:2], 1, 1)

In [134]:
sigma_x_high_threshold = 40
sigma_z_high_threshold = 150

em_sub = decode_emitters[
    (decode_emitters.xyz_sig_nm[:, 0] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 1] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 2] <= sigma_z_high_threshold)
    #* (decode_emitters.bg >= bg_trheshold)
    ]

decode_points = get_np_points(em_sub, 0, 100)
print(decode_points.shape)
viewer.add_points(decode_points, name='decode filtered', face_color='red', symbol='+',size=2)

(17284, 3)


<Points layer 'decode filtered' at 0x1cb25c99b20>

In [113]:

decode_points = get_np_points(decode_emitters, 0, 100)

decode_points[:,0] = decode_points[:,0] - 1 

viewer.add_points(decode_points, name='decode points', face_color='cyan', size=2, symbol='o')

<Points layer 'decode points [1]' at 0x1cb25ef8c10>

In [84]:
start_frame

161250

In [85]:
hesslab_emitters.frame_ix = hesslab_emitters.frame_ix - start_frame

In [86]:
frames.shape
backup = hesslab_emitters.clone()

In [99]:
hesslab_emitters = backup.clone()
#reverse_emitters(hesslab_emitters, frames.shape[1:], 0, 0)

In [104]:

hesslab_points = get_np_points(hesslab_emitters, 0, 100)
#reverse_emitters(hesslab_emitters, frames.shape[1:], 0, 2)
hesslab_points[:, [1,2]] = hesslab_points[:, [2,1]]  # swap x and y for napari
viewer.add_points(hesslab_points, name='hesslab points', face_color='blue', size=2, symbol='x')

<Points layer 'hesslab points' at 0x1cb10e6d6d0>

In [107]:
hesslab_emitters.prob.min(), hesslab_emitters.prob.max()

(tensor(1., dtype=torch.float64), tensor(1., dtype=torch.float64))

In [100]:
sigma_x_high_threshold = 100
sigma_y_high_threshold = 100
sigma_z_high_threshold = 200
prob_threshold = 0
decode_emitters_filtered = decode_emitters[
    (decode_emitters.xyz_sig_nm[:, 0] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 1] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 2] <= sigma_z_high_threshold)
    * (decode_emitters.prob >= prob_threshold)
    ]



In [101]:
len(decode_emitters), len(decode_emitters_filtered)

(874861, 533915)

In [95]:
decode_points_filtered = get_np_points(decode_emitters_filtered, 0, 10)
decode_points_filtered[:,0] = decode_points_filtered[:,0] - 1 
viewer.add_points(decode_points_filtered, name='decode points', face_color='green', size=5)

<Points layer 'decode points [1]' at 0x22c0854e520>

In [74]:
rendered_name = r'D:\Janelia_slm\PALM Acquisition 50Gb (depleted frames ie low density stochastic emitters)\crop1\out\network_CMOS_C13440_20CUc_rendered.tif'
rendered = imread(rendered_name)
rendered = rendered[:, ::-1, :]
viewer.add_image(rendered[:,:,::-1], name='rendered', scale = [1/13,1/13])

filtered_rendered_name = r'D:\Janelia_slm\PALM Acquisition 50Gb (depleted frames ie low density stochastic emitters)\crop1\out\network_CMOS_C13440_20CUc_filtered_rendered.tif'
filtered_rendered = imread(filtered_rendered_name)
filtered_rendered = filtered_rendered[:, ::-1, :]
viewer.add_image(filtered_rendered[:,:,::-1], name='filtered_rendered', scale = [1/13,1/13])

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Janelia_slm\\PALM Acquisition 50Gb (depleted frames ie low density stochastic emitters)\\crop1\\out\\network_CMOS_C13440_20CUc_rendered.tif'

In [11]:
rendered.shape

(5629, 9360, 3)